In [1]:
import os
import glob
import numpy as np
import xarray as xr
# import dask
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# import matplotlib.cm as cm
# from matplotlib.colors import BoundaryNorm
# from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
# from matplotlib.figure import Figure
import pandas as pd
from datetime import datetime, timedelta

In [2]:
datadir_obs = '/gpfs/wolf/cli120/proj-shared/zfeng/cacti/csapr/stats/'
datafile_obs = f'{datadir_obs}trackstats_20181015.0000_20190303.0000.nc'

root_datadir_m = '/gpfs/wolf/cli120/proj-shared/zfeng/cacti/les/'
datadir_m3 = f'{root_datadir_m}/20181129/gefs00/d3/stats/'
datafile_m3 = sorted(glob.glob(f'{datadir_m3}trackstats_20*'))
datafile_m3

out_dir = '/gpfs/wolf/cli120/proj-shared/zfeng/cacti/csapr/stats4lasso/'
out_basename = 'csapr_trackstats_'

# Hour window to extend beyond the model start/end time
hour_window = 3

In [3]:
ds_obs = xr.open_dataset(datafile_obs, mask_and_scale=True)
ds_obs

<xarray.Dataset>
Dimensions:                  (tracks: 6892, times: 60)
Coordinates:
  * tracks                   (tracks) int64 0 1 2 3 4 ... 6888 6889 6890 6891
  * times                    (times) int64 0 1 2 3 4 5 6 ... 54 55 56 57 58 59
Data variables: (12/37)
    track_duration           (tracks) int32 ...
    base_time                (tracks, times) datetime64[ns] ...
    meanlat                  (tracks, times) float32 ...
    meanlon                  (tracks, times) float32 ...
    area                     (tracks, times) float32 ...
    cloudnumber              (tracks, times) float64 ...
    ...                       ...
    start_split_tracknumber  (tracks) float64 ...
    start_split_timeindex    (tracks) float64 ...
    start_split_cloudnumber  (tracks) float64 ...
    end_merge_tracknumber    (tracks) float64 ...
    end_merge_timeindex      (tracks) float64 ...
    end_merge_cloudnumber    (tracks) float64 ...
Attributes:
    Title:                 Statistics of each track
    Institution:           Pacific Northwest National Laboratory
    Contact:               Zhe Feng, zhe.feng@pnnl.gov
    Created_on:            Mon Jan 31 14:49:58 2022
    startdate:             20181015.0000
    enddate:               20190303.0000
    timegap_hour:          0.5
    time_resolution_hour:  0.25
    pixel_radius_km:       0.5

In [4]:
ds_m3 = xr.open_mfdataset(datafile_m3, concat_dim='tracks', combine='nested').load()
ds_m3

<xarray.Dataset>
Dimensions:                  (tracks: 368, times: 60)
Coordinates:
  * tracks                   (tracks) int64 0 1 2 3 4 5 ... 363 364 365 366 367
  * times                    (times) int64 0 1 2 3 4 5 6 ... 54 55 56 57 58 59
Data variables: (12/37)
    track_duration           (tracks) int32 3 4 3 14 2 6 36 2 ... 2 1 1 2 1 1 1
    base_time                (tracks, times) datetime64[ns] 2018-11-29T12:00:...
    meanlat                  (tracks, times) float32 -30.4 -30.45 ... nan nan
    meanlon                  (tracks, times) float32 -63.47 -63.35 ... nan nan
    area                     (tracks, times) float32 18.0 20.75 14.5 ... nan nan
    cloudnumber              (tracks, times) float64 1.0 1.0 1.0 ... nan nan nan
    ...                       ...
    start_split_tracknumber  (tracks) float64 nan nan nan ... 292.0 350.0 305.0
    start_split_timeindex    (tracks) float64 nan nan nan nan ... 7.0 1.0 6.0
    start_split_cloudnumber  (tracks) float64 nan nan nan nan ... 4.0 9.0 21.0
    end_merge_tracknumber    (tracks) float64 nan nan nan 65.0 ... nan nan nan
    end_merge_timeindex      (tracks) float64 nan nan nan 6.0 ... nan nan nan
    end_merge_cloudnumber    (tracks) float64 nan nan nan 1.0 ... nan nan nan
Attributes:
    Title:                 Statistics of each track
    Institution:           Pacific Northwest National Laboratory
    Contact:               Zhe Feng, zhe.feng@pnnl.gov
    Created_on:            Thu Jun  9 19:36:53 2022
    startdate:             20181129.1200
    enddate:               20181130.0000
    timegap_hour:          0.5
    time_resolution_hour:  0.25
    pixel_radius_km:       0.5

In [24]:
# # Get min/max datetime from model track stats
# sdate = np.min(ds_m3.start_basetime.data)
# edate = np.max(ds_m3.end_basetime.data)
# sdate, edate

In [25]:
# sdate - np.timedelta64(6, 'h')

In [26]:
# sdate_str = pd.to_datetime(str(sdate_ext)).strftime('%Y%m%d.%H%M')
# edate_str = pd.to_datetime(str(edate_ext)).strftime('%Y%m%d.%H%M')
# sdate_str, edate_str

In [27]:
startdate = ds_m3.attrs['startdate']
enddate = ds_m3.attrs['enddate']
sdate = pd.to_datetime(f'{startdate[0:4]}-{startdate[4:6]}-{startdate[6:8]}T{startdate[9:11]}:{startdate[11:13]}')
edate = pd.to_datetime(f'{enddate[0:4]}-{enddate[4:6]}-{enddate[6:8]}T{enddate[9:11]}:{enddate[11:13]}')
sdate, edate

(Timestamp('2018-11-29 12:00:00'), Timestamp('2018-11-30 00:00:00'))

In [28]:
# Subset CSAPR track stats by time window
sdate_ext = sdate - pd.DateOffset(hours=hour_window)
edate_ext = edate + pd.DateOffset(hours=hour_window)
sdate_str = sdate_ext.strftime('%Y%m%d.%H%M')
edate_str = edate_ext.strftime('%Y%m%d.%H%M')
sdate_str, edate_str
ds_out = ds_obs.where((ds_obs.start_basetime >= sdate_ext) & (ds_obs.end_basetime <= edate_ext), drop=True)

In [29]:
# # Subset CSAPR track stats by time window
# sdate_ext = sdate - np.timedelta64(hour_window, 'h')
# edate_ext = edate + np.timedelta64(hour_window, 'h')
# ds_out = ds_obs.where((ds_obs.start_basetime >= sdate_ext) & (ds_obs.end_basetime <= edate_ext), drop=True)

In [30]:
# Update attributes
ds_out.attrs['startdate'] = sdate_str
ds_out.attrs['enddate'] = edate_str
ds_out.attrs['hour_window'] = hour_window

In [33]:
# Write output
out_filename = f'{out_dir}{out_basename}{sdate_str}_{edate_str}.nc'
ds_out.to_netcdf(out_filename, mode="w", format="NETCDF4", unlimited_dims="tracks",)
print(out_filename)

/gpfs/wolf/cli120/proj-shared/zfeng/cacti/csapr/stats4lasso/csapr_trackstats_20181129.0900_20181130.0300.nc
